# CyberSight DW — Exploratory Data Analysis

Basic EDA on the CICIDS 2017 dataset. This notebook loads a sample CSV file and explores its structure, distributions, and data quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('ggplot')

## 1. Load Sample Data

Load one CSV file from the CICIDS 2017 dataset.

In [ ]:
DATA_PATH = '../data/cicids2017/Friday-WorkingHours.pcap_ISCX.csv'

df = pd.read_csv(DATA_PATH, encoding='utf-8', low_memory=False)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

## 2. Column Types and Summary

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 3. Label Distribution

What types of traffic (attack vs benign) are in this file?

In [ ]:
label_counts = df['Label'].value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(10, 5))
label_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Label Distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Protocol Distribution

In [ ]:
protocol_map = {6: 'TCP', 17: 'UDP', 0: 'ICMP'}
df['Protocol_Name'] = df['Protocol'].map(protocol_map).fillna('OTHER')

proto_counts = df['Protocol_Name'].value_counts()
print(proto_counts)

fig, ax = plt.subplots(figsize=(6, 6))
proto_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%')
ax.set_ylabel('')
ax.set_title('Protocol Distribution')
plt.show()

## 5. Null / Missing Values

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_summary[null_summary['nulls'] > 0]

## 6. Infinity Values Check

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = {}
for col in numeric_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf

if inf_counts:
    print('Columns with Infinity values:')
    for col, cnt in inf_counts.items():
        print(f'  {col}: {cnt}')
else:
    print('No Infinity values found.')

## 7. Timestamp Range

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')

print(f'Earliest: {df["Timestamp"].min()}')
print(f'Latest:   {df["Timestamp"].max()}')
print(f'Span:     {df["Timestamp"].max() - df["Timestamp"].min()}')

## 8. Top Source IPs

In [ ]:
attack_df = df[df['Label'] != 'BENIGN']
top_ips = attack_df['Source IP'].value_counts().head(10)
print(top_ips)

fig, ax = plt.subplots(figsize=(10, 5))
top_ips.plot(kind='barh', ax=ax, color='crimson')
ax.set_title('Top 10 Attacking Source IPs')
ax.set_xlabel('Attack Count')
plt.tight_layout()
plt.show()